# 01 — Real Earth observation and GeoAI agreement

This notebook inspects the provenance-validated 2021 Earth-observation bundle and its repeated grouped GeoAI evaluation. Sentinel-2 Level-2A reflectance is remotely sensed evidence; ESA WorldCover 2021 is a historic pseudo-label product, not independent field ground truth. Missing or incompatible real artifacts raise an error—synthetic data are never substituted.


## Load and validate the real artifact bundle

Run urbantwin eo-fetch and urbantwin geo-train before opening this notebook. Validation binds the metrics and split records to the EO stack, provenance, configuration, timestamps, and SHA-256 hashes.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from urbantwin.data.earth_observation import WORLDCOVER_CLASSES, compute_indices
from urbantwin.data.spatial_features import summarize
from urbantwin.visualization.report_writer import validate_geoai_artifacts

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Start Jupyter from the UrbanTwin-AI repository root.")

bundle = validate_geoai_artifacts(ROOT)
if bundle is None:
    raise FileNotFoundError(
        "Validated GeoAI artifacts are missing. Run eo-fetch and geo-train first."
    )

metrics = bundle["metrics"].copy()
split_metrics = bundle["split_metrics"].copy()
details = bundle["details"]
provenance = bundle["provenance"]

with np.load(ROOT / "data/real/eo_stack.npz", allow_pickle=False) as stack:
    bands = stack["bands"].copy()
    labels = stack["labels"].copy()
    valid_mask = stack["valid_mask"].copy()
    observation_count = stack["observation_count"].copy()


## Acquisition and projected-grid provenance

Scene cloud metadata is a catalogue-level descriptor; the retained-pixel mask is determined separately from the aligned Scene Classification Layer and finite reflectance bands.


In [ ]:
grid_summary = pd.Series(
    {
        "data kind": provenance["data_kind"],
        "CRS": provenance["crs"],
        "resolution (m)": provenance["configuration"]["resolution_m"],
        "grid width": provenance["width"],
        "grid height": provenance["height"],
        "grid pixels": provenance["grid_pixel_count"],
        "valid pixels": provenance["valid_pixel_count"],
        "valid coverage (%)": 100
        * provenance["valid_pixel_count"]
        / provenance["grid_pixel_count"],
        "composite scenes": len(provenance["sentinel_items"]),
        "WorldCover year": provenance["configuration"]["worldcover_year"],
    },
    name="value",
)
grid_summary.to_frame()


In [ ]:
scene_table = pd.DataFrame(provenance["sentinel_items"])[
    ["id", "datetime", "eo_cloud_cover_pct", "processing_baseline"]
].rename(
    columns={
        "id": "STAC item",
        "datetime": "acquired",
        "eo_cloud_cover_pct": "catalogue cloud cover (%)",
        "processing_baseline": "processing baseline",
    }
)
scene_table


## Spatial indicators and represented land cover

All summaries use the finite Sentinel-2 and supported-WorldCover overlap. NDVI and NDBI describe this AOI and acquisition composite only; they are not causal explanations of traffic or independent land-cover labels.


In [ ]:
spatial = summarize(
    bands,
    labels,
    float(provenance["configuration"]["resolution_m"]),
)
indicator_summary = pd.DataFrame(
    {
        "NDVI": spatial["ndvi"],
        "NDBI": spatial["ndbi"],
    }
).T
coverage_summary = pd.Series(
    {
        "summary scope": spatial["summary_scope"],
        "valid overlap pixels": spatial["valid_overlap_pixels"],
        "valid overlap (%)": spatial["valid_overlap_percent"],
        "represented area (km²)": spatial["represented_area_km2"],
        "pixels observed in all scenes (%)": 100
        * np.count_nonzero(observation_count == len(provenance["sentinel_items"]))
        / observation_count.size,
    },
    name="value",
)
coverage_summary.to_frame()


In [ ]:
landcover = (
    pd.Series(spatial["landcover_percentages"], name="represented area (%)")
    .sort_values(ascending=False)
    .to_frame()
)
landcover


In [ ]:
ndvi, ndbi = compute_indices(bands)
supported = np.isin(labels, sorted(WORLDCOVER_CLASSES))
analysis_mask = valid_mask & supported

figure, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, values, title, color in (
    (axes[0], ndvi[analysis_mask], "NDVI", "#3a7d44"),
    (axes[1], ndbi[analysis_mask], "NDBI", "#9c6644"),
):
    axis.hist(values[np.isfinite(values)], bins=40, color=color, alpha=0.85)
    axis.set(title=title, xlabel="Index value", ylabel="Valid pixels")
    axis.grid(alpha=0.2)
figure.suptitle("Spectral-index distributions over the validated overlap")
figure.tight_layout()
plt.show()


## Repeated grouped GeoAI evaluation

The table and figure report agreement with WorldCover pseudo-labels across repeated group-disjoint splits. Macro-F1 exposes minority-class performance more clearly than overall agreement or weighted-F1.


In [ ]:
metric_view = metrics[
    [
        "method",
        "features",
        "split_count",
        "macro_f1",
        "macro_f1_std",
        "weighted_f1",
        "weighted_f1_std",
        "accuracy_vs_worldcover",
        "accuracy_vs_worldcover_std",
    ]
].sort_values("macro_f1", ascending=False)
metric_view


In [ ]:
plot_rows = metric_view.reset_index(drop=True)
positions = np.arange(len(plot_rows))
labels_for_plot = (
    plot_rows["method"].str.replace("_", " ")
    + " / "
    + plot_rows["features"].str.replace("_", " ")
)
figure, axis = plt.subplots(figsize=(10, 4.5))
axis.bar(
    positions,
    plot_rows["macro_f1"],
    yerr=plot_rows["macro_f1_std"],
    color="#2b6f8a",
    capsize=4,
)
axis.set_xticks(positions, labels_for_plot, rotation=25, ha="right")
axis.set(ylabel="Macro-F1", title="Grouped agreement with WorldCover pseudo-labels")
axis.grid(axis="y", alpha=0.2)
figure.tight_layout()
plt.show()


In [ ]:
paired_ablation = split_metrics.pivot(
    index=["method", "split_seed"],
    columns="features",
    values="macro_f1",
).reset_index()
required_features = {"bands_only", "bands_plus_indices"}
if not required_features.issubset(paired_ablation.columns):
    raise ValueError("GeoAI split metrics do not contain the declared feature ablation.")
paired_ablation["macro_f1_delta_indices_minus_bands"] = (
    paired_ablation["bands_plus_indices"] - paired_ablation["bands_only"]
)
ablation_summary = (
    paired_ablation.groupby("method")["macro_f1_delta_indices_minus_bands"]
    .agg(["mean", "std", "min", "max", "count"])
    .sort_index()
)
ablation_summary


In [ ]:
support_diagnostics = pd.DataFrame(details["split_diagnostics"])[
    [
        "seed",
        "status",
        "eligible_class_ids",
        "unsupported_test_class_support",
        "low_test_support_diagnostic",
    ]
].copy()
support_diagnostics["eligible_classes"] = support_diagnostics[
    "eligible_class_ids"
].map(lambda values: [WORLDCOVER_CLASSES[int(value)] for value in values])
support_diagnostics.drop(columns="eligible_class_ids")


The ablation is paired by split seed, but only three spatial splits were evaluated. Its deltas are descriptive—not a significance test or evidence that spectral indices causally improve classification. Eligible classes are selected from training support only; test-only unsupported classes remain documented in the validated details bundle.
